[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](
https://colab.research.google.com/github/Simone-Alghisi/HMD-Lab/blob/master/notebooks/7_rag.ipynb)

On Colab:
1. Switch to a GPU Runtime by clicking on *Runtime > Change runtime type > T4 GPU*
2. Run the cell below

In [ ]:
# For Google Colab only
!git clone https://github.com/Simone-Alghisi/HMD-Lab.git
%cd /content/HMD-Lab/notebooks 

# Integrating External Knowledge

LLMs are static models: their knowledge is frozen to the (pre-)training stage.

However, some facts can change across time:
- Cristiano Ronaldo may play in a new football team
- Stocks for a given company may change
- The price for the pizza we serve may increase or decrease by a few dollars

<details>
    <summary> When users ask questions, the answer should be up-to-date. How can we solve this issue?
    </summary>
    <ul>
        <li> <b>Fine-Tune the model on new information</b>: costly, sometimes the model "forgets" (i.e., replaces useful information) something else, sometimes does not replace information </li>
        <li> <b>Retrieve external knowledge and use it to condition the generation</b>: where do we take it from, how do we keep it outdated, can we ensure that the system will use it? </li>
    </ul>
</details>



## Question
1. Which knowledge source do you know?
2. How can we access them?

## Structured Knowledge Sources

Databases and API are considered structured knowledge sources.

PROS:
- we can access their information through a query
- if information exists, we are guaranteed to access it

CONS:
- most of the information is not stored in knowledge bases
- it is costly and time-consuming to keep these sources up-to-date
- may require a paid subscription to access the content

How can we use this information with LLMs?

In [1]:
import sys

sys.path.append("..")

from utils import MODELS
from transformers import AutoTokenizer

model_name, InitModel, prepare_text = MODELS["qwen3"]

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = InitModel(
    model_name,
    dtype="auto",
    device_map="cuda:0",
)

/home/simone/miniconda3/envs/hmd/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 3/3 [00:01<00:00,  1.98it/s]



In [2]:
import torch

from notebooks.notebook_utils import display_conversation
from models.qwen3 import prepare_text

task_prompt = """You are given the Next Best Action (NBA), the Dialogue State (DS), and External Knowledge (EK).
The Next Best Action is a compact, machine-readable representation of what the dialogue manager wants to do next in the conversation.
The Dialogue State contains information about the user's intent and the extracted slot-value pairs:
{
    "intent": "...", 
    "slots": {
        "slot": "value"
    }
}
The External Knowledge provides relevant information to assist in generating the response, such as the total cost.

Based on Next Best Action, Dialogue State, and External Knowledge, generate a natural language response that is polite, concise, and contextually appropriate.
Output at most 50 words.
"""

In [3]:
nlg_input = """NBA: confirmation(pizza_ordering)
DS: {
  "intent": "pizza_ordering",
  "slots": {
    "pizza_size": "medium",
    "pizza_type": "margherita",
    "pizza_count": 2
  }
}
EK: total_cost: $25.00
"""

messages = [
    {
        "role": "system", 
        "content": task_prompt
    }
]

text = prepare_text(nlg_input, tokenizer, messages)

model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

with torch.no_grad():
    # max_new_tokens limits the length of the generated response
    generated_ids = model.generate(**model_inputs, max_new_tokens=50).cpu()

# decode the output
output_ids = generated_ids[0][len(model_inputs.input_ids[0]) :].tolist()
content = tokenizer.decode(output_ids, skip_special_tokens=True)
display_conversation(messages, nlg_input, content)


### Conversation

**System:** You are given the Next Best Action (NBA), the Dialogue State (DS), and External Knowledge (EK).
The Next Best Action is a compact, machine-readable representation of what the dialogue manager wants to do next in the conversation.
The Dialogue State contains information about the user's intent and the extracted slot-value pairs:
{
    "intent": "...", 
    "slots": {
        "slot": "value"
    }
}
The External Knowledge provides relevant information to assist in generating the response, such as the total cost.

Based on Next Best Action, Dialogue State, and External Knowledge, generate a natural language response that is polite, concise, and contextually appropriate.
Output at most 50 words.


**User:** NBA: confirmation(pizza_ordering)
DS: {
  "intent": "pizza_ordering",
  "slots": {
    "pizza_size": "medium",
    "pizza_type": "margherita",
    "pizza_count": 2
  }
}
EK: total_cost: $25.00


**Assistant:** Your order for 2 medium margherita pizzas is confirmed! Total cost: $25.00. Enjoy your pizza!

## Unstructured Knowledge Sources

However, most of the knowledge on the web is unstructured, in the form of documents/corpus in natural language and HTML/Markdown Tables.

PROS:
- (most) information can be freely accessed 
- important information is updated quickly (e.g., newspapers)

CONS:
- finding the right piece of information is a sea of documents is difficult
- up-to-date and updated documents may co-exist
- incoherent and fake information make the search even more complex

But, we can always decide where to take information from. Suppose that we have a set of reliable sources, how can we look for the right information among our documents?

### Retrieval-Augmented Generation (RAG)

Retrieval-Augmented Generation (RAG) combines two components:

- A **Retriever**, which selects relevant pieces of knowledge (documents, passages) based on a user query.
- A **Natural Language Generation (NLG)** component, which conditions the generation on the retrieved knowledge.

Workflow:
1. Encodes the user query and all the documents in our knowledge base using the Retriever
2. Select the top-k passages (e.g., documents, paragraphs, or sentences) with the highest similarity (e.g., cosine similarity, dot product, or euclidean disstance) with the query
3. Build a prompt that includes the top-k passages and the user question.
4. Pass it to the NLG to produce an answer

Easy, right? Let's see this in practice using [LangChain](https://docs.langchain.com/oss/python/langchain/rag)

In [ ]:
!pip install -q --upgrade langchain langchain-community faiss-cpu sentence-transformers langchain-huggingface langchain-text-splitters

In [4]:
from langchain_core.documents.base import Document

def build_mock_documents():
    texts = [
        "LangChain is a framework for developing applications powered by language models. It focuses on modular components like prompts, LLMs, memory, and chains, and makes it easier to build systems like RAG, agents, and chatbots.",
        "Retrieval-Augmented Generation (RAG) combines information retrieval with text generation. Documents are retrieved from a knowledge base and used to enrich the model's context, improving factual accuracy.",
        "Hugging Face provides tools such as the Transformers library and model hub. Many RAG systems rely on Hugging Face models for both embeddings and generation.",
        "FAISS is a library for efficient similarity search and clustering of dense vectors. It is commonly used as a vector database in RAG setups.",
    ]
    return [Document(page_content=t.strip()) for t in texts]

docs = build_mock_documents()
docs


[Document(metadata={}, page_content='LangChain is a framework for developing applications powered by language models. It focuses on modular components like prompts, LLMs, memory, and chains, and makes it easier to build systems like RAG, agents, and chatbots.'),
 Document(metadata={}, page_content="Retrieval-Augmented Generation (RAG) combines information retrieval with text generation. Documents are retrieved from a knowledge base and used to enrich the model's context, improving factual accuracy."),
 Document(metadata={}, page_content='Hugging Face provides tools such as the Transformers library and model hub. Many RAG systems rely on Hugging Face models for both embeddings and generation.'),
 Document(metadata={}, page_content='FAISS is a library for efficient similarity search and clustering of dense vectors. It is commonly used as a vector database in RAG setups.')]

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Split docs
splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=50,
)
split_docs = splitter.split_documents(docs)

# Embeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Vector store
vectorstore = FAISS.from_documents(split_docs, embeddings)

vectorstore


In [6]:
def retrieve_top_k(query, k=3):
    """
    Takes a text query, returns top-k relevant chunks (Documents)
    using the FAISS vector store.
    """
    results = vectorstore.similarity_search(query, k=k)
    return results

# Test retrieval alone
query = "What is RAG?"

# Change k to adjust number of docs retrieved
retrieved_docs = retrieve_top_k(query)

for i, doc in enumerate(retrieved_docs, 1):
    print(f"\n=== Retrieved doc {i} ===\n{doc.page_content[:300]}")



=== Retrieved doc 1 ===
Hugging Face provides tools such as the Transformers library and model hub. Many RAG systems rely on Hugging Face models for both embeddings and generation.

=== Retrieved doc 2 ===
Retrieval-Augmented Generation (RAG) combines information retrieval with text generation. Documents are retrieved from a knowledge base and used to enrich the model's context, improving factual accuracy.

=== Retrieved doc 3 ===
FAISS is a library for efficient similarity search and clustering of dense vectors. It is commonly used as a vector database in RAG setups.


In [ ]:
def build_context(docs):
    """
    Join retrieved documents into a single context string.
    """
    return "\n\n---\n\n".join(d.page_content for d in docs)

task_prompt = """You are a helpful question-answering assistant.
Use the context provided to answer the question as accurately as possible.
If the answer is not contained in the context, say you are not sure.
"""

def answer_question(query, k=3):
    # 1) retrieve
    retrieved_docs = retrieve_top_k(query, k=k)
    context = build_context(retrieved_docs)

    # 2) build prompt
    messages = [
        {
            "role": "system", 
            "content": task_prompt
        }
    ]
    
    user_input = f"Context: {context}\n\nQuestion: {query}"
    text = prepare_text(user_input, tokenizer, messages)

    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    # 3) generate
    with torch.no_grad():
        generated_ids = model.generate(**model_inputs, max_new_tokens=50).cpu()

    # decode the output
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]) :].tolist()
    content = tokenizer.decode(output_ids, skip_special_tokens=True)
    display_conversation(messages, user_input, content)


# Try it
answer_question("What is Retrieval-Augmented Generation (RAG)?", k=3)


### Conversation

**System:** You are a helpful question-answering assistant.
Use the context provided to answer the question as accurately as possible.
If the answer is not contained in the context, say you are not sure.


**User:** Context: Retrieval-Augmented Generation (RAG) combines information retrieval with text generation. Documents are retrieved from a knowledge base and used to enrich the model's context, improving factual accuracy.

---

Hugging Face provides tools such as the Transformers library and model hub. Many RAG systems rely on Hugging Face models for both embeddings and generation.

---

FAISS is a library for efficient similarity search and clustering of dense vectors. It is commonly used as a vector database in RAG setups.

Question: What is Retrieval-Augmented Generation (RAG)?

**Assistant:** Retrieval-Augmented Generation (RAG) is a technique that combines information retrieval with text generation. Documents are retrieved from a knowledge base and used to enrich the model's context, improving factual accuracy.

In [9]:
answer_question("What is the best brand of coffee?", k=3)

### Conversation

**System:** You are a helpful question-answering assistant.
Use the context provided to answer the question as accurately as possible.
If the answer is not contained in the context, say you are not sure.


**User:** Context: LangChain is a framework for developing applications powered by language models. It focuses on modular components like prompts, LLMs, memory, and chains, and makes it easier to build systems like RAG, agents, and chatbots.

---

FAISS is a library for efficient similarity search and clustering of dense vectors. It is commonly used as a vector database in RAG setups.

---

Retrieval-Augmented Generation (RAG) combines information retrieval with text generation. Documents are retrieved from a knowledge base and used to enrich the model's context, improving factual accuracy.

Question: What is the best brand of coffee?

**Assistant:** The context provided does not contain information about the best brand of coffee. Therefore, I am not sure.

## References

- [DyKnow: Dynamically Verifying Time-Sensitive Factual Knowledge in LLMs.](https://aclanthology.org/2024.findings-emnlp.471) In Findings of the Association for Computational Linguistics: EMNLP 2024, pages 8014–8029, Miami, Florida, USA. Association for Computational Linguistics.
- [LangChain](https://www.langchain.com)